# 02 — Cleaning and features

Notebook 01 produced one row per pair with everything that was known before the date. This one turns
that into a table a model can actually be fitted on: drop what is unusable, build features that
compare the two profiles rather than describe them separately, and encode the categories properly.

The last section checks, with numbers, that the way we split the data changes the result enough to
matter.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)

pairs = pd.read_csv("data/processed/pairs.csv")
print(pairs.shape, "| match rate:", pairs.match.mean().round(3))

(4184, 138) | match rate: 0.165


## 1. What we drop

Three reasons to drop a column: it is free text with hundreds of values, it is too incomplete to be
worth imputing, or it duplicates something we already have in a cleaner form.

In [2]:
FREE_TEXT = ["field", "undergra", "from", "zipcode", "career"]   # coded versions exist for two of them
TOO_EMPTY = ["expnum", "mn_sat", "tuition"]                      # 57% to 79% missing

# `4_1` (what you think your peers look for) was not asked in waves 1-5, `5_1` (how you think others
# see you) was skipped by 41% of participants.
WEAK_BLOCKS = [f"{q}{b}" for b in ["4_1", "5_1"] for q in ["attr", "sinc", "intel", "fun", "amb", "shar"]]

drop_per_person = FREE_TEXT + TOO_EMPTY + WEAK_BLOCKS
dropped = [f"{side}_{c}" for side in ["w", "m"] for c in drop_per_person if f"{side}_{c}" in pairs.columns]

df = pairs.drop(columns=dropped).copy()
print("dropped", len(dropped), "columns ->", df.shape)

dropped 38 columns -> (4184, 100)


### Before dropping the text: is there anything in it?

`field` and `career` are only a messier spelling of `field_cd` and `career_c` — "Law", "law" and
"Law and English Literature (J.D./Ph.D.)" are all code 1, and 167 spellings for 551 people would give
us one dummy per person. But `from`, `undergra` and `zipcode` have no coded equivalent, and coming
from the same town or the same university is a plausible reason to get on. That is worth measuring
before throwing it away.

In [3]:
def normalise(s):
    return s.str.lower().str.strip().str.replace(r"[^a-z0-9 ]", "", regex=True)

for name, col in [("same origin", "from"), ("same university", "undergra"), ("same postcode", "zipcode")]:
    a, b = normalise(pairs[f"w_{col}"]), normalise(pairs[f"m_{col}"])
    same = (a == b) & a.notna() & b.notna()
    print(f"{name:16s} {same.sum():4d} pairs ({same.mean():5.1%})   match rate "
          f"{pairs.match[same].mean():.3f} against {pairs.match[~same].mean():.3f}")

same origin        46 pairs ( 1.1%)   match rate 0.196 against 0.165
same university     3 pairs ( 0.1%)   match rate 0.000 against 0.165
same postcode       7 pairs ( 0.2%)   match rate 0.000 against 0.165


Three pairs went to the same university and seven grew up in the same postcode. The idea was
reasonable and the data does not support it: the event drew people from all over, so almost nobody
shares an exact origin with their date, and nothing can be learnt from three pairs. The columns go,
not because they are text but because they are empty of usable signal here.

What we keep instead is `same_career`, built further down from the coded version: it covers 18% of
the pairs, which is enough to measure.

## 2. Missing income is a signal, not a hole

Notebook 01 showed that income is missing for 62% of Latino and 60% of Asian participants against 31%
of Black participants, because the column is empty for people who grew up abroad. Imputing it would
erase that, so we keep the value and add an explicit flag saying it was absent.

In [4]:
for side in ["w", "m"]:
    df[f"{side}_income_missing"] = df[f"{side}_income"].isna().astype(int)

print(df[["w_income_missing", "m_income_missing"]].mean().round(3))

w_income_missing    0.406
m_income_missing    0.573
dtype: float64


## 3. Features that compare the two profiles

This is the part that matters. A model fed with "her age" and "his age" has to learn the notion of an
age gap on its own, from 4,184 rows. Handing it the gap directly is both easier to fit and easier to
explain to the client.

In [5]:
# Built into their own frame and joined back at the end of the section, so that pandas does not
# have to rewrite the table on every assignment.
new = pd.DataFrame(index=df.index)

new["age_gap"] = (df.w_age - df.m_age).abs()
new["age_diff"] = df.w_age - df.m_age          # signed: is she older than him?

In [6]:
QUALITIES5 = ["attr", "sinc", "intel", "fun", "amb"]   # `3_1` has no `shar`, so we use the five shared ones

# What she says she is looking for (100 points) against how he rates himself (1-10), and the reverse.
new["w_gets_what_she_wants"] = sum(df[f"w_{q}1_1"] * df[f"m_{q}3_1"] for q in QUALITIES5) / 100
new["m_gets_what_he_wants"] = sum(df[f"m_{q}1_1"] * df[f"w_{q}3_1"] for q in QUALITIES5) / 100

new[["w_gets_what_she_wants", "m_gets_what_he_wants"]].describe().round(2)

,w_gets_what_she_wants,m_gets_what_he_wants
count,4087.00,4079.00
mean,6.78,7.00
std,0.99,0.92
min,2.90,3.34
25%,6.17,6.40
50%,6.80,7.00
75%,7.45,7.60
max,10.50,11.85


In [7]:
ACTIVITIES = ["sports", "tvsports", "exercise", "dining", "museums", "art", "hiking", "gaming",
              "clubbing", "reading", "tv", "theater", "movies", "concerts", "music", "shopping", "yoga"]

# `int_corr` already measures how similar their interests are overall. A count of the things they are
# both enthusiastic about says something different: shared enthusiasm rather than a matching shape.
new["shared_passions"] = sum(((df[f"w_{a}"] >= 8) & (df[f"m_{a}"] >= 8)).astype(int) for a in ACTIVITIES)

print(df.match.groupby(new.shared_passions).agg(["mean", "size"]).round(3).head(8))

                  mean  size
shared_passions             
0                0.141   562
1                0.151   728
2                0.166   818
3                0.173   734
4                0.175   525
5                0.176   324
6                0.165   231
7                0.171   146


In [8]:
# Same background, same plans, same rhythm.
new["same_field"] = (df.w_field_cd == df.m_field_cd).astype(int)
new["same_career"] = (df.w_career_c == df.m_career_c).astype(int)
new["same_goal"] = (df.w_goal == df.m_goal).astype(int)
new["go_out_gap"] = (df.w_go_out - df.m_go_out).abs()
new["date_gap"] = (df.w_date - df.m_date).abs()

# How much either of them cares about dating inside their own group.
new["imprace_max"] = df[["w_imprace", "m_imprace"]].max(axis=1)
new["imprelig_max"] = df[["w_imprelig", "m_imprelig"]].max(axis=1)

`w_round` and `m_round` already hold the number of partners each of them met that evening, which is
the "how much choice did they have" effect measured in notebook 01. It has a direct equivalent for the
client: how many profiles the app puts in front of a user. We keep both and add the pair's average.

In [9]:
new["pool_size"] = (df.w_round + df.m_round) / 2       # replaces w_round and m_round
new["date_order"] = (df.w_order + df.m_order) / 2     # replaces w_order and m_order

df = pd.concat([df, new], axis=1)
print("built", new.shape[1], "comparison features ->", df.shape)

built 14 comparison features -> (4184, 116)


In [10]:
# How each of them moves the match rate on its own, as a first sanity check.
for c in ["age_gap", "shared_passions", "same_field", "same_goal", "imprace_max"]:
    q = pd.qcut(df[c], 3, labels=["low", "mid", "high"], duplicates="drop") if df[c].nunique() > 3 else df[c]
    print(f"{c:16s}", df.groupby(q, observed=True).match.mean().round(3).to_dict())

age_gap          {'low': 0.189, 'mid': 0.163, 'high': 0.132}
shared_passions  {'low': 0.154, 'mid': 0.173, 'high': 0.177}
same_field       {0: 0.157, 1: 0.229}
same_goal        {0: 0.165, 1: 0.164}
imprace_max      {'low': 0.194, 'mid': 0.159, 'high': 0.138}


An age gap costs about six points of match rate, sharing a field of study is worth seven, and caring
about dating inside your own ethnic group costs six. Sharing the same goal for the evening changes
nothing at all. None of them is decisive on its own, which is what we already expected.

## 4. Categories

`field_cd`, `career_c`, `goal` and `race` are labels stored as numbers, so they have to be one-hot
encoded. Several of their categories are tiny, and a dummy built on twenty people is noise we would
then have to interpret, so anything under 3% of the pairs goes into an "other" bucket.

In [11]:
def group_rare(s, min_share=0.03):
    share = s.value_counts(normalize=True)
    common = share[share >= min_share].index
    # `s.isna()` keeps the missing values missing: they must not land in the same bucket as the
    # rare categories, or "no answer" and "rare answer" become the same column.
    return s.where(s.isin(common) | s.isna(), other=-1)

CATEGORICAL = ["field_cd", "goal", "race"]   # `career_c` is covered by same_career and overlaps field

for side in ["w", "m"]:
    for c in CATEGORICAL:
        df[f"{side}_{c}"] = group_rare(df[f"{side}_{c}"])

print({c: sorted(int(v) for v in df[f"w_{c}"].dropna().unique()) for c in CATEGORICAL})

{'field_cd': [-1, 1, 3, 5, 6, 7, 8, 9, 10, 11, 13, 15], 'goal': [1, 2, 3, 4, 5, 6], 'race': [1, 2, 3, 4, 6]}


In [12]:
dummies = pd.get_dummies(df[[f"{s}_{c}" for s in ["w", "m"] for c in CATEGORICAL]].astype("category"),
                         prefix_sep="=", dummy_na=False, dtype=int)

model_df = pd.concat([df.drop(columns=[f"{s}_{c}" for s in ["w", "m"] for c in CATEGORICAL]), dummies],
                     axis=1)
print("dummies added:", dummies.shape[1], "-> table", model_df.shape)

dummies added: 42 -> table (4184, 152)


## 5. Features that say the same thing twice

Duplicated information does not make a model better, it makes it harder to read: a linear model
splits one coefficient between the copies, and a tree model splits the importance. So before fixing
the table, we look for columns that move together.

In [13]:
Xc = model_df.drop(columns=["pair", "wave", "match"]).select_dtypes("number")
corr = Xc.corr().abs().to_numpy().copy()
np.fill_diagonal(corr, 0)
corr = pd.DataFrame(corr, index=Xc.columns, columns=Xc.columns)

top = corr.where(np.triu(np.ones(corr.shape), 1).astype(bool)).stack().sort_values(ascending=False)
print(top[top > 0.75].round(3).to_string())

m_order     date_order      0.997
w_order     date_order      0.997
m_round     pool_size       0.991
w_round     pool_size       0.990
w_order     m_order         0.989
w_round     m_round         0.964
m_museums   m_art           0.862
w_museums   w_art           0.837
w_imprelig  imprelig_max    0.780
w_imprace   imprace_max     0.754


`pool_size` and `date_order` are averages of the two columns they came from, so they repeat them
almost exactly: we keep the average and drop the originals. `imprace_max` and `imprelig_max` sit
around 0.75 with the two columns they are the maximum of, which is expected and low enough to leave
alone.

In [14]:
NEAR_DUPLICATES = ["w_round", "m_round", "w_order", "m_order"]
model_df = model_df.drop(columns=NEAR_DUPLICATES)
print(model_df.shape)

(4184, 148)


Three larger blocks are redundant in the same way, for reasons that come from the questionnaire
rather than from arithmetic. `career_c` overlaps heavily with the field of study and is already
summarised by `same_career`. The `2_1` block records what each person *thinks* the opposite sex
wants, which is a belief about other people rather than a fact about them. And the seventeen activity
ratings are already condensed into `int_corr` and `shared_passions`.

Rather than argue about it, we fit the model both ways.

In [15]:
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from xgboost import XGBClassifier

REDUNDANT = ([f"{s}_career_c" for s in ["w", "m"]]
             + [f"{s}_{q}2_1" for s in ["w", "m"] for q in ["attr", "sinc", "intel", "fun", "amb", "shar"]]
             + [f"{s}_{a}" for s in ["w", "m"] for a in ACTIVITIES])

model = make_pipeline(SimpleImputer(strategy="median"),
                      XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                                    subsample=0.8, colsample_bytree=0.8, eval_metric="logloss"))
cv = StratifiedGroupKFold(5, shuffle=True, random_state=0)

full = model_df.drop(columns=["pair", "wave", "match"])
for name, Xi in [("everything", full), ("without the three blocks", full.drop(columns=REDUNDANT))]:
    auc = cross_val_score(model, Xi, model_df.match, cv=cv, groups=model_df.wave, scoring="roc_auc")
    print(f"{name:26s} {Xi.shape[1]:3d} features   ROC-AUC {auc.mean():.3f}")

everything                 145 features   ROC-AUC 0.580


without the three blocks    97 features   ROC-AUC 0.598


The shorter table is not worse, it is marginally better, and 104 features on 4,184 rows is a much
healthier ratio than 152. It is also easier to explain and cheaper for the tabular foundation model,
which does not like very wide inputs. We drop them.

In [16]:
model_df = model_df.drop(columns=REDUNDANT)
print(model_df.shape)

(4184, 100)


## 6. The table we will model on

`pair`, `wave` and `match` are not features: `pair` identifies the row, `wave` is the key we group on
when splitting, and `match` is the target. Everything else is a feature.

In [17]:
NOT_FEATURES = ["pair", "wave", "match"]

X = model_df.drop(columns=NOT_FEATURES)
y = model_df.match
groups = model_df.wave

print("features:", X.shape[1], "| rows:", len(X), "| positives:", round(y.mean(), 3))
print("still missing, worst five:")
print(X.isna().mean().sort_values(ascending=False).head(5).round(3))

features: 97 | rows: 4184 | positives: 0.165
still missing, worst five:
m_income                 0.573
w_income                 0.406
m_gets_what_he_wants     0.025
w_gets_what_she_wants    0.023
date_gap                 0.023
dtype: float64


We leave the remaining gaps as they are. XGBoost handles them natively, and for the models that cannot
we impute inside the pipeline, on the training fold only — imputing here would let the test fold's
values leak into the training set through the median.

In [18]:
model_df.to_csv("data/processed/model_table.csv", index=False)
print("saved", model_df.shape, "to data/processed/model_table.csv")

saved (4184, 100) to data/processed/model_table.csv


## 7. Does the split really matter?

Notebook 01 showed that every participant appears in about sixteen pairs and belongs to exactly one
wave. So a random split scatters the same person across both sides, while grouping by wave keeps them
together. The difference is not a detail.

In [19]:
from sklearn.model_selection import KFold, GroupKFold

schemes = {"random": (KFold(5, shuffle=True, random_state=0), None),
           "by wave": (GroupKFold(5), groups),
           "by wave, balanced": (StratifiedGroupKFold(5, shuffle=True, random_state=0), groups)}

rows = []
for name, (cv, g) in schemes.items():
    auc = cross_val_score(model, X, y, cv=cv, groups=g, scoring="roc_auc")
    ap = cross_val_score(model, X, y, cv=cv, groups=g, scoring="average_precision")
    rows.append({"split": name, "ROC-AUC": auc.mean().round(3), "spread": (auc.max()-auc.min()).round(3),
                 "PR-AUC": ap.mean().round(3)})

pd.DataFrame(rows).set_index("split")

,ROC-AUC,spread,PR-AUC
split,,,
random,0.720,0.039,0.362
by wave,0.584,0.092,0.218
"by wave, balanced",0.598,0.107,0.216


The random split looks much better than the grouped one, and it is an illusion: the model has already
seen a dozen of each person's other dates and has learnt that this particular profile says yes a lot.
Individual match rates run from 0% to 80%, so knowing *who* is on the row is worth a great deal — and
in production the app faces people it has never seen.

The two grouped versions tell the same story. We keep the balanced one because it guarantees that
every fold holds a comparable share of matches, which is what makes comparing folds meaningful at
all — a plain group split can hand one fold an evening where almost nobody matched.

What neither version fixes is the spread: seven to ten points of AUC between the best and the worst
fold, on a mean of 0.60. The waves really are different evenings, and one number will not describe
this model honestly. That spread is not noise to average away, it is the first thing the stability
section has to explain.

## What the next notebook gets

- `data/processed/model_table.csv`: one row per pair, the target, the wave, and the features.
- Split with `StratifiedGroupKFold(5)` grouped on `wave`, always.
- Ethnicity is still in the table on both sides, which the fairness notebook needs: it will compare a
  model fitted with it against one fitted without, and check whether `imprace`, income and the rest
  put the gap back on their own.